# Ejercicios ensembling
En este ejercicio vas a realizar prediciones sobre un dataset de ciudadanos indios diabéticos. Se trata de un problema de clasificación en el que intentaremos predecir 1 (diabético) 0 (no diabético).

### 1. Carga las librerias que consideres comunes al notebook

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import VotingClassifier
from sklearn.linear_model import LogisticRegression

### 2. Lee los datos de [esta direccion](https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv)
Los nombres de columnas son:
```Python
names = ['preg', 'plas', 'pres', 'skin', 'test', 'mass', 'pedi', 'age', 'class']
```

In [2]:
datos = pd.read_csv ('https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv')

In [3]:
names = ['preg', 'plas', 'pres', 'skin', 'test', 'mass', 'pedi', 'age', 'class']
datos.columns = names

In [4]:
datos.info()

<class 'pandas.DataFrame'>
RangeIndex: 767 entries, 0 to 766
Data columns (total 9 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   preg    767 non-null    int64  
 1   plas    767 non-null    int64  
 2   pres    767 non-null    int64  
 3   skin    767 non-null    int64  
 4   test    767 non-null    int64  
 5   mass    767 non-null    float64
 6   pedi    767 non-null    float64
 7   age     767 non-null    int64  
 8   class   767 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB


In [10]:
datos.head()

,preg,plas,pres,skin,test,mass,pedi,age,class
0,1,85,66,29,0,26.6,0.351,31,0
1,8,183,64,0,0,23.3,0.672,32,1
2,1,89,66,23,94,28.1,0.167,21,0
3,0,137,40,35,168,43.1,2.288,33,1
4,5,116,74,0,0,25.6,0.201,30,0


In [5]:
datos.describe()

,preg,plas,pres,skin,test,mass,pedi,age,class
count,767.000000,767.000000,767.000000,767.000000,767.000000,767.000000,767.000000,767.000000,767.000000
mean,3.842243,120.859192,69.101695,20.517601,79.903520,31.990482,0.471674,33.219035,0.348110
std,3.370877,31.978468,19.368155,15.954059,115.283105,7.889091,0.331497,11.752296,0.476682
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.078000,21.000000,0.000000
25%,1.000000,99.000000,62.000000,0.000000,0.000000,27.300000,0.243500,24.000000,0.000000
50%,3.000000,117.000000,72.000000,23.000000,32.000000,32.000000,0.371000,29.000000,0.000000
75%,6.000000,140.000000,80.000000,32.000000,127.500000,36.600000,0.625000,41.000000,1.000000
max,17.000000,199.000000,122.000000,99.000000,846.000000,67.100000,2.420000,81.000000,1.000000


In [ ]:
#Ya se ve que dependiendo del modelo habría que usar un esacalado porque los pesos son muy diferentes
# En tst por ejemplo tiene un std mucho mayor que el resto: tiene

### 3. Bagging
Para este apartado tendrás que crear un ensemble utilizando la técnica de bagging ([BaggingClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.BaggingClassifier.html)), mediante la cual combinarás 100 [DecisionTreeClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html). Recuerda utilizar también [cross validation](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.KFold.html) con 10 kfolds.

**Para este apartado y siguientes, no hace falta que dividas en train/test**, por hacerlo más sencillo. Simplemente divide tus datos en features y target.

Establece una semilla

In [ ]:
# si pongo menos árboles tiene sentido poner más complejidad en cada árbol
# n_estimators y max_samples no están relacionados directemtne pero puede tener sentido tocar uno si toco el otro
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import KFold, cross_val_score

X = datos[['preg', 'plas', 'pres', 'skin', 'test', 'mass', 'pedi', 'age']]
y = datos ['class']

model = DecisionTreeClassifier(max_depth=3,random_state=42)

bag_clf = BaggingClassifier(
    estimator = model,
    n_estimators=100, # Cantidad de árboles
    random_state=42,
    )


kfold = KFold(n_splits=10) #Parte los datos en 10 trozos para usar validación cruzada / cross validation
cv_results_bag = cross_val_score(bag_clf, X, y, cv=kfold, scoring='accuracy')
print(cv_results_bag)
print('Media scores;', cv_results_bag.mean())
print('Desviacion:', cv_results_bag.std())

[0.7012987  0.79220779 0.75324675 0.62337662 0.77922078 0.81818182
 0.83116883 0.82894737 0.72368421 0.73684211]
Media scores; 0.7588174982911824
Desviacion: 0.062136288707606915


### 4. Random Forest
En este caso entrena un [RandomForestClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html) con 100 árboles y un `max_features` de 3. También con validación cruzada

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rnd_clf = RandomForestClassifier(n_estimators=100,
                                 max_features=3,
                                 random_state=42)

kfold = KFold(n_splits=10) #Parte los datos en 10 trozos para usar validación cruzada / cross validation
# si a cv abajo le pongo cv=10 funciona también, no hay que ponerle nada
cv_results_ranf = cross_val_score(rnd_clf, X, y, cv=kfold, scoring='accuracy')
print(cv_results_ranf)
print('Media scores;', cv_results_ranf.mean())
print('Desviacion:', cv_results_ranf.std())
#Este modelo es más sensible al anterior a los outliers

[0.68831169 0.81818182 0.74025974 0.62337662 0.81818182 0.81818182
 0.81818182 0.85526316 0.69736842 0.77631579]
Media scores; 0.7653622693096378
Desviacion: 0.07121220709488776


### 5. AdaBoost
Implementa un [AdaBoostClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.AdaBoostClassifier.html) con 30 árboles.

In [37]:
from sklearn.ensemble import AdaBoostClassifier

model = DecisionTreeClassifier (max_depth=1)
ada_clf = AdaBoostClassifier(estimator = model,
                             n_estimators=30,
                             learning_rate=0.5, #lo he puesto así porque es la medida que pusimos en clase. si se quita y se deja por defecto empeora un poquito
                             random_state=42)

# no lo dice, pero lo entreno también con cross-validation como los anteriores

kfold = KFold(n_splits=10) #Parte los datos en 10 trozos para usar validación cruzada / cross validation
cv_results_ada = cross_val_score(ada_clf, X, y, cv=kfold, scoring='accuracy')
print(cv_results_ada)
print('Media scores;', cv_results_ada.mean())
print('Desviacion:', cv_results_ada.std())

[0.67532468 0.80519481 0.7012987  0.67532468 0.76623377 0.81818182
 0.80519481 0.82894737 0.76315789 0.77631579]
Media scores; 0.7615174299384827
Desviacion: 0.05504696112400708


### 6. GradientBoosting
Implementa un [GradientBoostingClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.GradientBoostingClassifier.html) con 100 estimadores

In [38]:
from sklearn.ensemble import GradientBoostingClassifier

gbct = GradientBoostingClassifier(max_depth=2,
                                 n_estimators=100,
                                 learning_rate=1.0,
                                 random_state=42)

kfold = KFold(n_splits=10) #Parte los datos en 10 trozos para usar validación cruzada / cross validation
cv_results_gb = cross_val_score(gbct, X, y, cv=kfold, scoring='accuracy')
print(cv_results_gb)
print('Media scores;', cv_results_gb.mean())
print('Desviacion:', cv_results_gb.std())

[0.67532468 0.76623377 0.63636364 0.67532468 0.77922078 0.72727273
 0.77922078 0.86842105 0.68421053 0.77631579]
Media scores; 0.7367908407382091
Desviacion: 0.06622182856515212


### 7. XGBoost
Para este apartado utiliza un [XGBoostClassifier](https://docs.getml.com/latest/api/getml.predictors.XGBoostClassifier.html) con 100 estimadores. XGBoost no forma parte de la suite de modelos de sklearn, por lo que tendrás que instalarlo con pip install

In [39]:
import xgboost
xgb_reg = xgboost.XGBClassifier(random_state=42)

kfold = KFold(n_splits=10) #Parte los datos en 10 trozos para usar validación cruzada / cross validation
cv_results_xb = cross_val_score(xgb_reg, X, y, cv=kfold, scoring='accuracy')
print(cv_results_xb)
print('Media scores;', cv_results_xb.mean())
print('Desviacion:', cv_results_xb.std())


[0.67532468 0.79220779 0.68831169 0.62337662 0.77922078 0.72727273
 0.72727273 0.77631579 0.67105263 0.76315789]
Media scores; 0.7223513328776487
Desviacion: 0.053420586807056775


### 8. Primeros resultados
Crea un dataframe con los resultados y sus algoritmos, ordenándolos de mayor a menor

In [41]:
resultados = {
    'Algoritmo': ['Bagging', 'Random Forest','Gradient Boosting', 'AdaBoost', 'XGBoost'],
    'Media': [cv_results_bag.mean(), cv_results_ranf.mean(), cv_results_gb.mean(), cv_results_ada.mean(), cv_results_xb.mean()],
    'Desviacion': [cv_results_bag.std(), cv_results_ranf.std(), cv_results_gb.std(), cv_results_ada.std(), cv_results_xb.std()]
}

df_resultados = pd.DataFrame(resultados)
df_resultados = df_resultados.sort_values('Media', ascending=False)
df_resultados

,Algoritmo,Media,Desviacion
1,Random Forest,0.765362,0.071212
3,AdaBoost,0.761517,0.055047
0,Bagging,0.758817,0.062136
2,Gradient Boosting,0.736791,0.066222
4,XGBoost,0.722351,0.053421


### 9. Hiperparametrización
Vuelve a entrenar los modelos de nuevo, pero esta vez dividiendo el conjunto de datos en train/test y utilizando un gridsearch para encontrar los mejores hiperparámetros.

In [42]:
from sklearn.model_selection import train_test_split, GridSearchCV

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [43]:

# Bagging. Modifico solo los parámetros que le indicamos antes
param_grid_bag = {
    'n_estimators': [50, 100, 200],
    'estimator__max_depth': [1, 2, 3]
}
bag_base = BaggingClassifier(estimator=DecisionTreeClassifier(), random_state=42)
bag_best = GridSearchCV(bag_base, param_grid_bag, cv=10, scoring='accuracy')
bag_best.fit(X_train, y_train)

print("Bagging - Mejores params:", bag_best.best_params_)
print("Bagging - Accuracy test:", bag_best.score(X_test, y_test))


Bagging - Mejores params: {'estimator__max_depth': 3, 'n_estimators': 100}
Bagging - Accuracy test: 0.7922077922077922


In [44]:

# Random Forest
param_grid_rf = {
    'n_estimators': [50, 100, 200],
    'max_features': [2, 3, 5]
}
rf_best = GridSearchCV(RandomForestClassifier(random_state=42), param_grid_rf, cv=10, scoring='accuracy')
rf_best.fit(X_train, y_train)
print("Random Forest - Mejores params:", rf_best.best_params_)
print("Random Forest - Accuracy test:", rf_best.score(X_test, y_test))


Random Forest - Mejores params: {'max_features': 2, 'n_estimators': 50}
Random Forest - Accuracy test: 0.8051948051948052


In [45]:

# AdaBoost
param_grid_ada = {
    'n_estimators': [20, 30, 50],
    'learning_rate': [0.1, 0.5, 1.0]
}
best_ada = GridSearchCV(AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=1), random_state=42), 
                      param_grid_ada, cv=10, scoring='accuracy')
best_ada.fit(X_train, y_train)

print("AdaBoost - Mejores params:", best_ada.best_params_)
print("AdaBoost - Accuracy test:", best_ada.score(X_test, y_test))


AdaBoost - Mejores params: {'learning_rate': 0.5, 'n_estimators': 50}
AdaBoost - Accuracy test: 0.7922077922077922


In [48]:
# Gradient boosting
param_grid_gb = {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.1, 0.5, 1.0],
    'max_depth': [2, 3, 5]
}
best_gb = GridSearchCV(GradientBoostingClassifier(random_state=42), param_grid_gb, cv=10, scoring='accuracy')
best_gb.fit(X_train, y_train)
print("Gradient Boosting - Mejores params:", best_gb.best_params_)
print("Gradient Boosting - Accuracy test:", best_gb.score(X_test, y_test))


Gradient Boosting - Mejores params: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 100}
Gradient Boosting - Accuracy test: 0.7792207792207793


In [49]:
# XGBoost - los mismos que para gradient

param_grid_xgb = {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.01, 0.1, 0.3],
    'max_depth': [3, 5, 7]
}
best_xgb= GridSearchCV(xgboost.XGBClassifier(random_state=42), param_grid_xgb, cv=10, scoring='accuracy')
best_xgb.fit(X_train, y_train)
print("XGBoost - Mejores params:", best_xgb.best_params_)
print("XGBoost - Accuracy test:", best_xgb.score(X_test, y_test))

XGBoost - Mejores params: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 50}
XGBoost - Accuracy test: 0.7857142857142857


In [50]:
# Me hago el dataframe como antes para verlo
resultados_gs = {
    'Algoritmo': ['Bagging', 'Random Forest', 'AdaBoost', 'Gradient Boosting', 'XGBoost'],
    'Mejores params': [bag_best.best_params_, rf_best.best_params_, best_ada.best_params_, best_gb.best_params_, best_xgb.best_params_],
    'Accuracy test': [bag_best.score(X_test, y_test), rf_best.score(X_test, y_test), best_ada.score(X_test, y_test), 
                      best_gb.score(X_test, y_test), best_xgb.score(X_test, y_test)]
}

df_resultados_gs = pd.DataFrame(resultados_gs)
df_resultados_gs = df_resultados_gs.sort_values('Accuracy test', ascending=False)
df_resultados_gs

,Algoritmo,Mejores params,Accuracy test
1,Random Forest,"{'max_features': 2, 'n_estimators': 50}",0.805195
0,Bagging,"{'estimator__max_depth': 3, 'n_estimators': 100}",0.792208
2,AdaBoost,"{'learning_rate': 0.5, 'n_estimators': 50}",0.792208
4,XGBoost,"{'learning_rate': 0.1, 'max_depth': 3, 'n_esti...",0.785714
3,Gradient Boosting,"{'learning_rate': 0.1, 'max_depth': 3, 'n_esti...",0.779221


In [51]:
resultados_gs['Mejores params']

[{'estimator__max_depth': 3, 'n_estimators': 100},
 {'max_features': 2, 'n_estimators': 50},
 {'learning_rate': 0.5, 'n_estimators': 50},
 {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 100},
 {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 50}]

### 10. Conclusiones finales

Los modelos han mejorado todos cuando hemos hiperparametrizado. Pero tanto en su versión base como después no hay grandes diferencias entre los distintos modelos (apenas 2 puntos porcentuales).
Llama la atención que siendo teóricamente los más potentes, GradientBoosting y XBoost son los que peor se comportan. Parece que es un problema demasiado sencillo para estos modelos.
El Random Forest es el que mejor lo resuelve tanto en el modelo base como hiperparametrizando